# Task 1 — Notebook 01: Data Ingestion (CDL + NDVI)

**Purpose (NAFSI Task 1):** Load **CDL** as a crop mask and **MODIS NDVI** time series for the Corn Belt (Iowa / Nebraska focus in config), compare corn vs. soybean phenology. Data are read from **interim NetCDF** produced by `scripts/build_interim_data.py` (already EPSG:5070).

**Inputs (interim):**  
- `data/interim/cdl/cdl_stack_{y0}_{y1}.nc` — all available CDL years in one stack (`year`, `y`, `x`)  
- `data/interim/ndvi/ndvi_weekly_{year}.nc` — one file per calendar year; combined here along `calendar_year`

**This notebook:** loads the **full** stacks, then selects the analysis year from `configs/task1_ndvi_analysis.yaml` for `cdl` and matching NDVI growing-season weeks. QA smoothing and exports happen in later notebooks.

In [ ]:
# NOTE: This notebook was scaffolded with AI assistance. All analysis logic
# and interpretation must be reviewed and authored by the team.

import sys
from pathlib import Path

import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root (requirements.txt + src/). cd to the repo and retry.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.interim_loaders import load_cdl_stack_from_interim, load_ndvi_weekly_all_years

with open(REPO_ROOT / "configs" / "task1_ndvi_analysis.yaml") as f:
    cfg = yaml.safe_load(f)

cdl_year = int(cfg["cdl"]["year"])
ndvi_year = int(cfg["ndvi"]["year"])
print("Config:", "CDL year", cdl_year, "| NDVI year", ndvi_year)

In [ ]:
# Full CDL stack from interim; slice analysis year for crop mask (corn / soybean).
cdl_stack = load_cdl_stack_from_interim(REPO_ROOT)
_years = {int(y) for y in cdl_stack["year"].values}
if cdl_year not in _years:
    raise ValueError(f"Config CDL year {cdl_year} not in interim stack years {sorted(_years)}")
cdl = cdl_stack.sel(year=cdl_year)
print("cdl_stack", dict(cdl_stack.sizes), "| cdl (mask year)", dict(cdl.sizes), cdl.dtype)


In [ ]:
# All NDVI weekly years in one array (calendar_year, time, y, x); slice analysis year.
ndvi_all = load_ndvi_weekly_all_years(REPO_ROOT)
_cy = {int(y) for y in ndvi_all["calendar_year"].values}
if ndvi_year not in _cy:
    raise ValueError(f"Config NDVI year {ndvi_year} not in interim calendar_year {sorted(_cy)}")
ndvi_year_da = ndvi_all.sel(calendar_year=ndvi_year).squeeze("calendar_year", drop=True)
ndvi = ndvi_year_da.rename("ndvi")
print("ndvi_all", dict(ndvi_all.sizes), "| ndvi (analysis year)", dict(ndvi.sizes))


In [ ]:
# Interim stacks match the download grid (CONUS Albers EPSG:5070 in the pipeline).
# Next notebooks: align CDL → NDVI resolution if needed, purity filter, QA, smoothing.

try:
    import rioxarray  # noqa: F401

    print("CDL CRS:", getattr(cdl.rio, "crs", None))
    print("NDVI CRS:", getattr(ndvi.rio, "crs", None))
except Exception as exc:
    print("CRS check skipped (optional rioxarray .rio):", exc)
